In [1]:
pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 13.1 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install duckdb pandas pyarrow neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 22.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [neo4j]32m1/2 [neo4j]


In [4]:
import pandas as pd
import duckdb
from pathlib import Path

In [5]:
# Main project folder
project = Path.home() / "Desktop" / "Big data and Bussiness intelligence Capstone Project"

# Subfolders
data = project / "data"
output = project / "output_files"

# Create folders
data.mkdir(parents=True, exist_ok=True)
output.mkdir(parents=True, exist_ok=True)

print("Folders created successfully!")
print(project)

Folders created successfully!
/Users/rudrapibgle/Desktop/Big data and Bussiness intelligence Capstone Project


In [6]:
DATA_DIR = Path.home() / "Desktop" / "Big data and Bussiness intelligence Capstone Project" / "data"

list(DATA_DIR.iterdir())

[PosixPath('/Users/rudrapibgle/Desktop/Big data and Bussiness intelligence Capstone Project/data/tokenized_access_logs.csv'),
 PosixPath('/Users/rudrapibgle/Desktop/Big data and Bussiness intelligence Capstone Project/data/DataCoSupplyChainDataset.csv'),
 PosixPath('/Users/rudrapibgle/Desktop/Big data and Bussiness intelligence Capstone Project/data/DescriptionDataCoSupplyChain.csv')]

In [9]:
PROJECT_DIR = Path.home() / "Desktop" / "Big data and Bussiness intelligence Capstone Project"
DATA_DIR = PROJECT_DIR / "data"

MAIN_FILE = DATA_DIR / "DataCoSupplyChainDataset.csv"
DB_FILE = PROJECT_DIR / "supplychain.duckdb"

con = duckdb.connect(str(DB_FILE))

con.execute(f"""
CREATE OR REPLACE TABLE supply_raw AS
SELECT *
FROM read_csv_auto(
    '{MAIN_FILE}',
    encoding='cp1252',
    header=True
);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
DESC_FILE = DATA_DIR / "DescriptionDataCoSupplyChain.csv"

con.execute(f"""
CREATE OR REPLACE TABLE description_raw AS
SELECT *
FROM read_csv_auto(
    '{DESC_FILE}',
    encoding='cp1252',
    header=True
);
""")

In [11]:
LOG_FILE = DATA_DIR / "tokenized_access_logs.csv"

con.execute(f"""
CREATE OR REPLACE TABLE logs_raw AS
SELECT *
FROM read_csv_auto(
    '{LOG_FILE}',
    encoding='cp1252',
    header=True
);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.execute("""
CREATE OR REPLACE TABLE clean_orders AS
SELECT DISTINCT
    "Order Id" AS order_id,
    "Product Card Id" AS product_id,
    "Product Name" AS product_name,
    "Product Price" AS product_price,
    "Order Region" AS region,
    "Order Country" AS country,
    "Shipping Mode" AS shipping_mode,
    CAST("Days for shipping (real)" AS DOUBLE) AS real_shipping_days,
    CAST("Days for shipment (scheduled)" AS DOUBLE) AS scheduled_shipping_days,
    CAST("Late_delivery_risk" AS INTEGER) AS late_flag,
    CAST("Sales" AS DOUBLE) AS sales,
    CAST("Order Profit Per Order" AS DOUBLE) AS profit
FROM supply_raw
WHERE "Product Name" IS NOT NULL
  AND "Order Region" IS NOT NULL
  AND "Shipping Mode" IS NOT NULL;
""")

In [13]:
con.execute("SELECT * FROM clean_orders LIMIT 5").df()

,order_id,product_id,product_name,product_price,region,country,shipping_mode,real_shipping_days,scheduled_shipping_days,late_flag,sales,profit
0,16395,403,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,Southern Europe,Italia,Standard Class,6.0,4.0,1,129.990005,-70.779999
1,10297,403,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,Northern Europe,Reino Unido,Standard Class,3.0,4.0,0,129.990005,0.000000
2,64647,403,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,Northern Europe,Reino Unido,Standard Class,3.0,4.0,0,129.990005,63.049999
3,11532,403,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,Western Europe,Francia,Standard Class,3.0,4.0,0,129.990005,40.950001
4,11016,502,Nike Men's Dri-FIT Victory Golf Polo,50.000000,Northern Europe,Reino Unido,Standard Class,2.0,4.0,0,50.000000,14.070000


In [14]:
con.execute("SELECT COUNT(*) AS total_rows FROM clean_orders").df()

,total_rows
0,180511
